# 02. Route Optimization, Comparison, and Sensitivity Analysis
This notebook demonstrates running the Easy (Dijkstra) and Hard (Cubic Spline SLSQP) route optimizers, calculating the Price of Realism, and analyzing sensitivity sweeps.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_config, fetch_and_reproject_gebco
from src.features import derive_features
from src.easy import solve_easy_problem
from src.hard import solve_hard_problem
from src.metrics import compare_easy_and_hard

cfg = load_config('../config.yaml')
data = fetch_and_reproject_gebco(cfg)
feat = derive_features(data, cfg)


In [ ]:
# Solve Easy Problem (Dijkstra on 8-neighbor graph)
easy_res, f_mask = solve_easy_problem(data, feat, cfg)
print(f"Easy Length: {easy_res['L_E']:.2f} m | Risk: {easy_res['S_E']:.4f} | Max Curvature: {easy_res['max_kappa_fit']:.6f} m^-1")


In [ ]:
# Solve Hard Problem (Multi-start SLSQP Cubic Spline)
hard_res = solve_hard_problem(data, feat, cfg, easy_res=easy_res)
print(f"Hard Length: {hard_res['L_H']:.2f} m | Risk: {hard_res['S_H']:.4f} | Max Curvature: {hard_res['kappa_max']:.6f} m^-1")
print(f"Strictly Feasible: {hard_res['is_feasible']}")


In [ ]:
# Benchmark Comparison Table
comp, df_comp = compare_easy_and_hard(easy_res, hard_res, cfg)
df_comp


In [ ]:
# Plot both routes on Risk Map
plt.figure(figsize=(9, 8))
plt.imshow(feat['R'], cmap='inferno', origin='upper')
plt.colorbar(label='Composite Risk R')

path_rc = np.array(easy_res['path_rc'])
plt.plot(path_rc[:, 1], path_rc[:, 0], 'c--', linewidth=2.5, label='Easy (Dijkstra)')

dx, dy = cfg['grid']['dx'], cfg['grid']['dy']
xmin, ymax = cfg['grid']['xmin'], cfg['grid']['ymax']
hx = (np.array(hard_res['path_x']) - xmin) / dx
hy = (ymax - np.array(hard_res['path_y'])) / dy
plt.plot(hx, hy, 'lime', linewidth=3.0, label='Hard (SLSQP Spline)')

plt.scatter([cfg['endpoints']['point_A']['grid_col']], [cfg['endpoints']['point_A']['grid_row']], c='white', s=120, label='Point A')
plt.scatter([cfg['endpoints']['point_B']['grid_col']], [cfg['endpoints']['point_B']['grid_row']], c='yellow', s=120, label='Point B')
plt.title(f"Optimal Alignments (Price of Realism = {comp['metrics']['price_of_realism_percent']:+.2f}%)", fontsize=12, fontweight='bold')
plt.legend(loc='lower left')
plt.show()
